# Segmentation performance by lesion size (public data)

Train two 48-tree pixel classifiers using RGB and local intensity features at 48 × 48 resolution. Compare uniform pixel sampling against equal foreground/background sampling. Each seed uses 400 training and 200 held-out images. Define small lesions using the median training-mask area and report image-level mean Dice in both strata. The source has 1,000 Kvasir-SEG endoscopy image/mask pairs. Each seed samples 600 without replacement. Splits are at image level. Size strata describe lesion geometry. Demographic fairness and patient-level generalization were outside this experiment.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'lesion-size-segmentation-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'Pillow==12.3.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('from pathlib import Path\nimport hashlib\nimport os\nimport urllib.request\nimport zipfile\n\nroot = Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\npath = root / \'kvasir-seg/kvasir-seg.zip\'\npath.parent.mkdir(parents=True, exist_ok=True)\nif not path.exists():\n    with urllib.request.urlopen(\'https://datasets.simula.no/downloads/kvasir-seg.zip\',timeout=120) as response:\n        payload = response.read()\n    if hashlib.sha256(payload).hexdigest() != \'03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7\':\n        raise ValueError("Downloaded release checksum mismatch")\n    path.write_bytes(payload)\nif hashlib.sha256(path.read_bytes()).hexdigest() != \'03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7\':\n    raise ValueError("Local dataset checksum mismatch")\nwith zipfile.ZipFile(path) as archive:\n    archive.extractall(path.parent / "extracted")\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.ensemble import ExtraTreesClassifier\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", str(ROOT / "data")))\nSEEDS = [13, 37, 61]\n\n\ndef save(model, name):\n    path = ROOT / "checkpoints"\n    path.mkdir(exist_ok=True)\n    joblib.dump(model, path / (name + ".joblib"), compress=3)\n\n\nfrom PIL import Image\n\n\ndef examples(seed):\n    rng = np.random.default_rng(seed)\n    base = DATA / "kvasir-seg" / "extracted" / "Kvasir-SEG"\n    files = sorted((base / "images").glob("*.jpg"))\n    assert len(files) >= 1000\n    ids = rng.choice(len(files), 600, replace=False)\n    x = []\n    y = []\n    for i in ids:\n        f = files[i]\n        with Image.open(f) as im:\n            x.append(np.array(im.convert("RGB").resize((48, 48))) / 255)\n        with Image.open(base / "masks" / f.name) as im:\n            y.append(\n                np.array(im.convert("L").resize((48, 48), Image.Resampling.NEAREST))\n                > 127\n            )\n    return np.array(x), np.array(y)\n\n\ndef features(im):\n    gray = im.mean(2)\n    return np.dstack(\n        [\n            im,\n            ndi.gaussian_filter(gray, 2),\n            ndi.gaussian_gradient_magnitude(gray, 1),\n            ndi.uniform_filter(gray, 9),\n        ]\n    ).reshape(-1, 6)\n\n\ndef run(seed):\n    x, y = examples(seed)\n    rng = np.random.default_rng(seed)\n    a = np.stack([features(img) for img in x])\n    sizes = y.mean((1, 2))\n    thresh = np.median(sizes[:400])\n    out = {}\n    for method in ["uniform", "balanced_foreground"]:\n        inputs = []\n        labels = []\n        for z, m in zip(a[:400], y[:400]):\n            flat = m.ravel()\n            idx = rng.choice(len(flat), 160)\n            if method == "balanced_foreground" and flat.any() and (~flat).any():\n                idx = np.r_[\n                    rng.choice(np.flatnonzero(flat), 80),\n                    rng.choice(np.flatnonzero(~flat), 80),\n                ]\n            inputs.append(z[idx])\n            labels.append(flat[idx])\n        model = ExtraTreesClassifier(\n            n_estimators=48, min_samples_leaf=5, n_jobs=2, random_state=seed\n        ).fit(np.vstack(inputs), np.concatenate(labels))\n        save(model, f"{method}-{seed}")\n        p = model.predict_proba(a[400:].reshape(-1, 6))[:, 1].reshape(-1, 48, 48)\n        pred = p >= 0.5\n        truth = y[400:]\n        dice = (2 * (pred & truth).sum((1, 2)) + 1) / (\n            pred.sum((1, 2)) + truth.sum((1, 2)) + 1\n        )\n        small = sizes[400:] <= thresh\n        out[method] = {\n            "small_dice": float(dice[small].mean()),\n            "large_dice": float(dice[~small].mean()),\n            "all_dice": float(dice.mean()),\n        }\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())